In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
from rapidfuzz import process
import pickle

  Obtaining dependency information for rapidfuzz from https://files.pythonhosted.org/packages/5e/00/861a4601e4685efd8161966cf35728806fb9df112b6951585bb194f74379/rapidfuzz-3.13.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.4 MB/s eta 0:00:00a 0:00:01
DEPRECATION: qiskit-nature 0.5.1 has a non-standard dependency specifier qiskit-terra>=0.22.*. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of qiskit-nature or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [3]:
# Count number of species per aviary 
metadata_path = 'metadata_aviaries/'
species_df = pd.read_csv(metadata_path + 'Avieries_obsolete.csv', encoding='latin-1')

number_species = species_df.groupby('Aviary')['Common name'].nunique()
number_individuals = species_df.groupby('Aviary')['Total count'].sum()
metadata_paths = species_df.groupby('Aviary')['preprocessed data'].first()  

combined_df = pd.DataFrame({'Number of Species': number_species, 'Number of Individuals': number_individuals, "preprocessed data": metadata_paths})
combined_df = combined_df.dropna() # Some aviaries don't have metadata files, so we drop those rows for now

# Link preprocessed data path to metadata path as they are not in the same format
aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
for idx, row in combined_df.iterrows():
    avirary_path = row["preprocessed data"]
    best_match = process.extractOne(avirary_path, aviaries_metadata)
    if best_match and best_match[1] > 80:
        file_path = best_match[0]

    combined_df.loc[row.name, "Metadata_filename"] = file_path

combined_df.reset_index(inplace=True)
combined_df


/tmp/ipykernel_6700/2565535384.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'fl_3s_avifauna_flamingos_sept25_data_meta.xlsx' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  combined_df.loc[row.name, "Metadata_filename"] = file_path


,Aviary,Number of Species,Number of Individuals,preprocessed data,Metadata_filename
0,"Avifauna, Cuba aviary",8,247,fl_avifauna_flamingos_sept2025_data,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
1,"Avifauna, Umvikeli aviary",13,71,fl_avifauna_vultures_sept2025_data,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
2,"Avifauna, storage",4,36,fl_avifauna_storage_sept2025_data,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
3,"Beekse Bergen, Aviary 2",1,2,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
4,"Beekse Bergen, Aviary 3",8,163,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
5,"Beekse Bergen, Aviary 4 (Africa)",8,182,fl_beekse_bergen_20250412,fl_beekse_bergen_20250412_meta.xlsx
6,"Beekse Bergen, aviary 5",2,113,fl_beekse_bergen_flaminogos_sept2025_data,fl_beekse_bergen_20250412_meta.xlsx
7,"Blijdorp, Flamingos",4,204,fl_blijdorp_flamingos_dec2025_data,fl_blijdorp_flamingos_dec2025_data_meta.xlsx
8,"Cologne, Flamingos",1,153,fl_cologne_zoo_flamingos_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx
9,"Cologne, Hippodom",17,98,fl_cologne_zoo_indoors_long_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx


In [5]:
# count number of bird call per aviary meta data file
# We don't have all the metadata files, so we can't get the info for all aviaries in the cell above

aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
EDA_df = pd.DataFrame(columns=["file_name", "call_count", "total_entries", "Number of Species", "Number of Individuals"])

for file in aviaries_metadata:
    df = pd.read_excel(metadata_path + file)

    call_count = df["fusion_model_prediction"].notnull().sum()
    
    number_species = combined_df.loc[combined_df["Metadata_filename"] == file, "Number of Species"].iloc[0] if file in combined_df["Metadata_filename"].values else 0
    number_individuals = combined_df.loc[combined_df["Metadata_filename"] == file, "Number of Individuals"].iloc[0] if file in combined_df["Metadata_filename"].values else 0
    
    new_row = {
        "file_name": file,
        "call_count": call_count,
        "total_entries": len(df),
        "Ratio": call_count / len(df),
        "Number_Species": number_species,
        "Number_Individuals": number_individuals
    }
    
    EDA_df.loc[len(EDA_df)] = new_row
    
EDA_df

,file_name,call_count,total_entries,Number of Species,Number of Individuals
0,fl_zoo_eindhoven_20250308_meta.xlsx,65440,74420,NaN,NaN
1,fl_zoo_eindhoven_20250426_meta.xlsx,82988,97864,NaN,NaN
2,fl_gaia_zoo_savannah_08aug25_data_meta.xlsx,108628,126165,NaN,NaN
3,fl_gaia_zoo_congo_15aug25_data_meta.xlsx,27147,42872,NaN,NaN
4,fl_gaia_zoo_taiga_18Jul25_data_meta.xlsx,26295,34307,NaN,NaN
5,fl_avifauna_vultures_prox_sept25_data_meta.xlsx,19985,28980,NaN,NaN
6,fl_zoo_eindhoven_20250503_meta.xlsx,16465,29901,NaN,NaN
7,fl_beekse_bergen_20250412_meta.xlsx,33816,50560,NaN,NaN
8,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx,131557,148163,NaN,NaN
9,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx,117833,167919,NaN,NaN


In [73]:
EDA_df.to_csv("Aviary_EDA.csv", index=False)